# Attention Profiling on GPU

## 1. Verify/Install Correct NSight Version

### Add CUDA binaries to PATH if needed

In [ ]:
import os

os.environ['PATH'] = os.pathsep.join([os.environ['PATH'], '/usr/local/cuda/bin/'])

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
!ncu --version

**I used:**

For RTX 2060 GPU:
* CUDA V13.2.51
* NSight 2026.1.0.0

For T4, A100, L4, H100, G4 GPU:
* CUDA V12.8.93
* NSight 2025.1.1

## 2. Testing Script

Make sure it works before running it all at once with NSight

### Setup

In [ ]:
!pip install numpy pandas torch

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

In [ ]:
# following FlashAttention-3 paper
def generate_matrix(shape, seed=None) -> np.ndarray:
    if seed is not None:
        np.random.seed(seed)
    # Base matrix from N(0, 1)
    base = np.random.normal(loc=0.0, scale=1.0, size=shape)
    # Bernoulli mask (0.001 probability of being 1)
    mask = np.random.binomial(n=1, p=0.001, size=shape)
    # Noise from N(0, 100)
    noise = np.random.normal(loc=0.0, scale=10.0, size=shape)
    # Final matrix: base + noise * mask
    return base + noise * mask

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
device

In [ ]:
def scaled_dot_product_attention(Q_np: np.ndarray, K_np: np.ndarray, V_np: np.ndarray, causal: bool) -> np.ndarray:
    # ensure matching dimensions of 4D tensors
    assert (len(Q_np.shape), len(K_np.shape), len(V_np.shape)) == (4, 4, 4)
    b, h, seq_q, d = Q_np.shape
    bk, hk, seq_k, dk = K_np.shape
    bv, hv, seq_v, dv = V_np.shape
    assert b == 1 and b == bk and b == bv
    assert h == 1 and h == hk and h == hv
    assert d == dk, "Q and K head dim must be equal"
    assert d == dv, f"Q ({d}) and V ({dv}) head dim must be equal"
    assert seq_k == seq_v, "K and V must have equal seq len"

    # use CUDA on GPU
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    Q_torch = torch.from_numpy(Q_np).to(device)
    K_torch = torch.from_numpy(K_np).to(device)
    V_torch = torch.from_numpy(V_np).to(device)

    #####
    # for Turing arch, cannot use FlashAttention2
    # for Ampere+ GPUs, use SDPBackend.FLASH_ATTENTION
    #####
    # with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
    #     O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch,
    #                                              attn_mask=None,  # no masking
    #                                              dropout_p=0.0,  # no dropout
    #                                              is_causal=causal)
    #####

    ##### Use torch profiler to see which CUDA kernel it is using for attention by default
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CUDA],
        record_shapes=True,
        with_stack=False
    ) as prof:
        O_torch = F.scaled_dot_product_attention(Q_torch, K_torch, V_torch, attn_mask=None, dropout_p=0.0, is_causal=causal)
        torch.cuda.synchronize()

    print(prof.key_averages().table(sort_by="cuda_time_total", max_name_column_width=200))
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    #####

    return O_torch.cpu().numpy()

In [ ]:
seq_q = seq_kv = 256
d = 64
seed = 42
causal = False

### Create numpy arrays

Must be `batch_size x num_heads x seq_len x head_dim` for memory-efficient attention.

If it is 2D (`seq_len x head_dim` only), torch will fall back to the Math implementation that has no fused operations.

In [ ]:
# Use FP16 for FA1 or MemEff Attention
# Ensure 4D with correct axes to match MemEff implementation
Q_np = generate_matrix((seq_q, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
K_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
V_np = generate_matrix((seq_kv, d), seed=seed).astype(np.float16)[np.newaxis, np.newaxis, :, :]
Q_np.shape, K_np.shape, V_np.shape

RTX 2060 and T4 (Turing) use CUDA Kernel:

```
fmha_cutlassF_f16_aligned_64x64_rf_sm75(PyTorchMemEffAttention::AttentionKernel<cutlass::half_t, cutlass::arch::Sm75, true, 64, 64, 64, true, true>::Params)
```

A100, L4, H100, G4:

Small seq len (256):
```
void pytorch_flash::flash_fwd_kernel<Flash_fwd_kernel_traits<64, 128, 128, 4, false, false, cutlass::half_t, Flash_kernel_traits<64, 128, 128, 4, cutlass::half_t> >, false, false, false, false, tru...
```

Larger seq len (512+):
```
void pytorch_flash::flash_fwd_splitkv_kernel<Flash_fwd_kernel_traits<64, 64, 256, 4, false, false, cutlass::half_t, Flash_kernel_traits<64, 64, 256, 4, cutlass::half_t> >, false, false, false, true...

void pytorch_flash::flash_fwd_splitkv_combine_kernel<Flash_fwd_kernel_traits<64, 64, 256, 4, false, false, cutlass::half_t, Flash_kernel_traits<64, 64, 256, 4, cutlass::half_t> >, 8, 1, true>(pytor...
```

Note: "G4" is **RTX Pro 6000 Blackwell Server Edition**

### Run the attention algorithm

In [ ]:
scaled_dot_product_attention(Q_np, K_np, V_np, causal)

FlashAttention currently supports:

Turing, Ampere, Ada, or Hopper GPUs (e.g., H100, A100, RTX 3090, T4, RTX 2080).
fp16 and bf16 (bf16 requires Ampere, Ada, or Hopper GPUs).
Head dimensions that are multiples of 8, up to 128 (e.g., 8, 16, 24, ..., 128). Head dim > 64 backward requires A100 or H100.

## 3. Run Testing Script with NSight

In [ ]:
# On local: '.venv/bin/python'
# On CoLab: 'python'
MY_PYTHON_PATH = '.venv/bin/python'

In [ ]:
# Make sure the python path is correct for the environment where the profiles will be run (local vs Colab)
_python_path_checked = False
assert _python_path_checked, "Manually verify that MY_PYTHON_PATH is set to the correct python executable!"

### Test the testing script before profiling

In [ ]:
# make sure that the correct kernel is being used (MEA vs FA1/FA2)
_kernel_checked = False
assert _kernel_checked, "Manually verify that the correct attention kernel is being used (MEA vs FA1/FA2) before running the profiles!"

In [ ]:
!{MY_PYTHON_PATH} testing_script.py

### Find the correct kernel and Number of Kernels to skip

It should be the same as the number of warmup kernels in the `testing_script` (e.g. 5)

In [ ]:
!ncu --print-summary per-kernel {MY_PYTHON_PATH} testing_script.py

### Profile the Kernel

#### Gets a lot of metrics that are irrelevant, just make sure it's hitting the correct kernel

In [ ]:
!ncu --set full --launch-skip 10 --launch-count 1 -f -o profile_test_full {MY_PYTHON_PATH} testing_script.py

#### Get the correct metrics to profile

All metrics can be found in the [Metrics Reference documentation](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference)

In [ ]:
# Find metrics that are available on the GPU we are using
!ncu --query-metrics --target-processes all 2>/dev/null | grep "cycles_active"

In [ ]:
s = """
1tex__cycles_active.avg,
dram__cycles_active.avg,
gpu__time_duration.sum,
l1tex__cycles_active.avg,
lts__cycles_active.avg,
sm__cycles_active.avg,
sm__cycles_elapsed.avg,
sm__inst_executed.sum,
sm__inst_executed_pipe_lsu.avg,
sm__pipe_alu_cycles_active.avg,
sm__pipe_aluheavy_cycles_active.avg,
sm__pipe_alulite_cycles_active.avg,
sm__pipe_fma_cycles_active.avg,
sm__pipe_fmaheavy_cycles_active.avg,
sm__pipe_fmalite_cycles_active.avg,
sm__pipe_fp16_cycles_active.avg,
sm__pipe_fp64_cycles_active.avg,
sm__pipe_lsu_cycles_active.avg,
sm__pipe_shared_cycles_active.avg,
sm__pipe_tc_cycles_active.avg,
sm__pipe_tensor_cycles_active.avg,
sm__sass_thread_inst_executed_op_fadd_pred_on.sum,
sm__sass_thread_inst_executed_op_fmul_pred_on.sum,
sm__sass_thread_inst_executed_op_ffma_pred_on.sum,
sm__sass_thread_inst_executed_op_hadd_pred_on.sum,
smsp__inst_executed_pipe_alu.sum,
smsp__inst_executed_pipe_fma.sum,
smsp__inst_executed_pipe_lsu.sum,
smsp__inst_executed_pipe_tensor.sum,
smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed,
smsp__inst_executed_pipe_xu.sum,
smsp__issue_active.avg.pct_of_peak_sustained_elapsed,
smsp__pcsamp_warps_issue_stalled_lg_throttle,
smsp__pcsamp_warps_issue_stalled_long_scoreboard,
smsp__pcsamp_warps_issue_stalled_math_pipe_throttle,
smsp__pcsamp_warps_issue_stalled_membar,
smsp__pcsamp_warps_issue_stalled_not_selected,
smsp__pcsamp_warps_issue_stalled_short_scoreboard,
smsp__warps_issue_stalled_long_scoreboard.avg,
smsp__warps_issue_stalled_math_pipe_throttle.avg,
smsp__warps_issue_stalled_mio_throttle.avg,
smsp__warps_issue_stalled_not_selected.avg,
smsp__warps_issue_stalled_short_scoreboard.avg,
smsp__warps_issue_stalled_wait.avg
"""

metrics_to_measure = s.replace("\n", "").split(",")
metrics_to_measure = [m.strip() for m in metrics_to_measure]
metrics_str = ','.join(metrics_to_measure)
print(metrics_str)


#### Construct commands with correct metrics and parameters

In [ ]:
import torch

_raw = torch.cuda.get_device_name(0).lower()
for _prefix in ["nvidia geforce ", "nvidia ", "geforce ", "tesla ", "quadro "]:
    if _raw.startswith(_prefix):
        _raw = _raw[len(_prefix):]
        break
gpu_name = _raw.replace(" ", "")
print(f"GPU: {torch.cuda.get_device_name(0)} -> slug: {gpu_name}")

In [ ]:
seq_lens = [2**i for i in range(8, 14)]  # 256 to 8192
head_dims = [64, 128]
num_runs = 3
warmup_runs = 10

In [ ]:
# Turing GPUs always use 1 kernel regardless of seq len (MEA)
# Later GPUs use 2 kernels for long seq len (FA2+)
MEA_GPUS = ['rtx2060', 't4']

In [ ]:
cmds = []

for seq_len in seq_lens:
    for head_dim in head_dims:
        config_dir = f"profiles/{gpu_name}/{seq_len}x{head_dim}"
        os.makedirs(config_dir, exist_ok=True)

        # seq_len=256 uses 1 kernel, seq_len>=512 uses splitkv (2 kernels)
        kernels_per_call = 1 if gpu_name in MEA_GPUS or seq_len <= 256 else 2
        print(f"{gpu_name} {seq_len}x{head_dim} uses {kernels_per_call} kernels per call")
        launch_skip = warmup_runs * kernels_per_call
        launch_count = kernels_per_call

        for run_idx in range(num_runs):
            output_name = f"{config_dir}/run{run_idx}"
            cmd = (
                f"ncu --metrics {metrics_str}"
                f" --launch-skip {launch_skip} --launch-count {launch_count}"
                f" -f -o {output_name}"
                f" {MY_PYTHON_PATH} testing_script.py"
                f" --seq_q {seq_len} --seq_kv {seq_len} --d {head_dim} --warmup {warmup_runs}"
            )
            cmds.append(cmd)
            print(f'{gpu_name} {seq_len}x{head_dim} run {run_idx}:')
            print(cmd + '\n')

#### Lock the GPU and Memory clocks for reproducibility

##### First, find the Frequencies to use

In [ ]:
# All supported clocks on this GPU - there are a lot
# Format:
"""
Supported Clocks
        Memory                                         : <memory clock speed 1> MHz
            Graphics                                   : <graphics clock speed compatible with memory clock speed 1> MHz
            Graphics                                   : <graphics clock speed compatible with memory clock speed 1> MHz
            Graphics                                   : <graphics clock speed compatible with memory clock speed 1> MHz
            ...
        Memory                                         : <memory clock speed 2> MHz
            Graphics                                   : <graphics clock speed compatible with memory clock speed 2> MHz
            Graphics                                   : <graphics clock speed compatible with memory clock speed 2> MHz
            ...
"""

!nvidia-smi -q -d SUPPORTED_CLOCKS

In [ ]:
# This gets just the min and max graphics/memory clock speeds
!nvidia-smi --query-gpu=gpu_name,clocks.gr,clocks.max.gr,clocks.mem,clocks.max.mem --format=csv

For example, on RTX 2060, there are 15-MHz steps from 300 to 2145 MHz available for the max memory speed, 7001 MHz.

We want a base clock value that is _not idle_ but _not boosted_. Default settings can be found on NVIDIA's website:

- [RTX 2060](https://www.nvidia.com/en-gb/geforce/graphics-cards/compare/?section=compare-20)
- [T4](https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/tesla-t4/t4-tensor-core-product-brief.pdf)
- [A100](https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/PB-10577-001_v02.pdf)
- [L4](https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/l4/PB-11316-001_v01.pdf)
- [H100](https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/h100/PB-11773-001_v01.pdf)
- [RTX 6000 Pro Server (G4)](https://resources.nvidia.com/en-us-rtx-pro-6000/rtx-pro-6000-server-brief)

However, these may vary from values if device is factory-overclocked in an add-in board.

| GPU Name | Base GPU Clock (MHz) | Boost GPU Clock (MHz) | Max Memory Clock (MHz) |
| -------- | -------------------- | --------------------- | ---------------------- |
| RTX 2060 | 1370                 | 1680                  | 7001                   |
| T4       | 585                  | 1590                  | 5001                   |
| A100     | 1065                 | 1410                  | 1512                   |
| L4       | 795                  | 2040                  | 6251                   |
| H100     | 1080                 | 1785                  | 2619                   |
| G4       | 1852                 | 2430                  | 12481                  |


**Actual Clock Values:**

(These are based on the supported clocks found above)

RTX 2060: GPU=1380, Mem=7001

T4:     GPU=???, Mem=???

A100:   GPU=???, Mem=???

L4:     GPU=???, Mem=???

H100:   GPU=???, Mem=???

G4:     GPU=???, Mem=???

In [ ]:
gpu_clock = 1380  # MHz
memory_clock = 7001  # MHz

In [ ]:
_clocks_checked = False
assert _clocks_checked, "Manually verify that the GPU and memory clock speeds are set to the desired values before running the profiles!"

##### Lock the clocks

**IMPORTANT:** Find values appropriate for specific GPU from the above section before running this!

If running on local, need to use sudo. Put the sudo password in a .env file.

If running on CoLab, already root.

In [ ]:
_check_need_sudo = False
assert _check_need_sudo, "Manually verify whether sudo is needed to set clocks on this machine before running the profiles!"

In [ ]:
NEED_SUDO = True

In [ ]:
if NEED_SUDO:  # local
    !pip install python-dotenv
    from dotenv import load_dotenv
    load_dotenv()
    sudo_pwd = os.getenv("SUDO_PWD")
    assert sudo_pwd is not None, "SUDO_PWD environment variable not set. Please set it in the .env file before running the profiles!"
    !echo {sudo_pwd} | sudo -S nvidia-smi -pm 1  # enable persistence mode to keep clocks from dropping when idle
    !echo {sudo_pwd} | sudo -S nvidia-smi --lock-gpu-clocks={gpu_clock},{gpu_clock}
    !echo {sudo_pwd} | sudo -S nvidia-smi --lock-memory-clocks={memory_clock},{memory_clock}
else:  # Colab - no sudo needed
    !sudo nvidia-smi -pm 1  # enable persistence mode to keep clocks from dropping when idle
    !sudo nvidia-smi --lock-gpu-clocks={gpu_clock},{gpu_clock}
    !sudo nvidia-smi --lock-memory-clocks={memory_clock},{memory_clock}

#### Run the profiler with the correct metrics

In [ ]:
for cmd in cmds:
    !{cmd}

##### Unlock the clocks

In [ ]:
if NEED_SUDO:
    !echo {sudo_pwd} | sudo -S nvidia-smi --reset-gpu-clocks
    !echo {sudo_pwd} | sudo -S nvidia-smi --reset-memory-clocks
else:
    !sudo nvidia-smi --reset-gpu-clocks
    !sudo nvidia-smi --reset-memory-clocks

### View Profile Results

Instructions and example code for profiler API found in the [Python Report Interface documentation](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html)

#### Setup package

In [ ]:
!python -m pip install jupyterlab-nvidia-nsight

Add the directory for the `ncu_report` package in NSight to the PYTHON PATH

In [ ]:
import subprocess

result = subprocess.run(['find', '/usr', '/opt', '-name', 'ncu_report*',
                         '-type', 'f'],
                        capture_output=True, text=True)

path = result.stdout.splitlines()[0][:-len('ncu_report.py')]
# print(path)

import sys

sys.path.append(path)
import ncu_report

#### Get the report info

In [ ]:
import pandas as pd


def get_report_metrics(
        report_name: str,
        metrics_names: list | None = None,
        gpu_name: str | None = None,
        seq_len: int | None = None,
        head_dim: int | None = None,
        run_idx: int | None = None,
) -> pd.DataFrame:
    """
    Returns a wide-format DataFrame with one row per (range, action).

    Columns:
        GpuName, SeqLen, HeadDim, Run, Range, Action  -- metadata
        <metric_name>, ...                             -- one column per metric

    If `metrics_names` is provided, only those metrics appear as columns.
    Otherwise all metrics in the report are included.
    """
    _context = ncu_report.load_report(report_name)
    rows = []
    for r in range(_context.num_ranges()):
        _range = _context.range_by_idx(r)
        for a in range(_range.num_actions()):
            _action = _range.action_by_idx(a)
            row = {
                'GpuName': gpu_name,
                'SeqLen': seq_len,
                'HeadDim': head_dim,
                'Run': run_idx,
                'Range': r,
                'Action': _action.name(),
            }
            _names = [name for name in _action.metric_names()
                      if (metrics_names is None or name in metrics_names)]
            for name in _names:
                metric = _action.metric_by_name(name)
                metric_str = metric.as_string()
                metric_float = metric.as_double()
                row[name] = metric_str if (metric_str is not None and metric_float != 0.0) else metric_float
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
dfs = []
for seq_len in seq_lens:
    for head_dim in head_dims:
        for run_idx in range(num_runs):
            df = get_report_metrics(
                f"profiles/{gpu_name}/{seq_len}x{head_dim}/run{run_idx}.ncu-rep",
                metrics_to_measure,
                gpu_name=gpu_name, seq_len=seq_len,
                head_dim=head_dim, run_idx=run_idx,
            )
            dfs.append(df)

all_metrics = pd.concat(dfs, ignore_index=True)
all_metrics.to_csv(f'metrics_{gpu_name}.csv', index=False)

##### Example report content

In [ ]:
REPORT_NAME = rf"profiles/{gpu_name}/2048x64/run0.ncu-rep"

my_context = ncu_report.load_report(REPORT_NAME)
my_context.num_ranges()

In [ ]:
my_range = my_context.range_by_idx(0)
my_range.num_actions()

In [ ]:
my_action = my_range.action_by_idx(0)
my_action.name()

In [ ]:
df = get_report_metrics(REPORT_NAME, metrics_to_measure)

# Filter to action of interest
row = df[df['Action'] == my_action.name()].iloc[0]

present = [c for c in metrics_to_measure if c in df.columns]
missing = [m for m in metrics_to_measure if m not in df.columns]

print(f"Missing metrics ({len(missing)}):")
for key in sorted(missing):
    print(f"  {key}")

print(f"\nAvailable metrics ({len(present)}):")
for key in sorted(present):
    print(f"  {key}: {row[key]}")

**TODO: UPDATE FOR NEW BLACKWELL METRICS**

RTX 2060 is missing: (UP TO DATE)

-  1tex__cycles_active.avg
-  sm__pipe_aluheavy_cycles_active.avg
-  sm__pipe_alulite_cycles_active.avg
-  sm__pipe_fmaheavy_cycles_active.avg
-  sm__pipe_fmalite_cycles_active.avg
-  sm__pipe_fp16_cycles_active.avg
-  sm__pipe_lsu_cycles_active.avg
-  sm__pipe_tc_cycles_active.avg

T4, A100 is missing:

-  1tex__cycles_active.avg
-  sm__pipe_aluheavy_cycles_active.avg
-  sm__pipe_alulite_cycles_active.avg
-  sm__pipe_fmaheavy_cycles_active.avg
-  sm__pipe_fmalite_cycles_active.avg
-  sm__pipe_fp16_cycles_active.avg
-  sm__pipe_lsu_cycles_active.avg

L4 is missing:

-  1tex__cycles_active.avg
-  sm__pipe_aluheavy_cycles_active.avg
-  sm__pipe_alulite_cycles_active.avg
-  sm__pipe_fp16_cycles_active.avg
-  sm__pipe_lsu_cycles_active.avg
-  sm__pipe_shared_cycles_active.avg

H100 is missing:
-  1tex__cycles_active.avg
-  sm__pipe_aluheavy_cycles_active.avg
-  sm__pipe_alulite_cycles_active.avg
-  sm__pipe_fp16_cycles_active.avg
-  sm__pipe_lsu_cycles_active.avg

G4 is missing:
-  1tex__cycles_active.avg
-  sm__pipe_alulite_cycles_active.avg
-  sm__pipe_fp16_cycles_active.avg
-  sm__pipe_lsu_cycles_active.avg
-  sm__pipe_shared_cycles_active.avg

### Zip profiles to download from CoLab

In [ ]:
!zip -r {gpu_name}.zip profiles/{gpu_name}

### Join all metrics

In [ ]:
from pathlib import Path
import pandas as pd

csv_files = sorted(Path("metrics").glob("*.csv"))
all_metrics = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
all_metrics.to_csv("metrics_all.csv", index=False)

print(f"Combined {len(csv_files)} files into metrics_all.csv")
all_metrics.shape